In [1]:
import os
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
from huggingface_hub import login
import gradio as gr

hf_token = os.getenv("HF_TOKEN")
login(hf_token, add_to_git_credential=True)

c:\projects\synth-gen-llm\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [ ]:
# model_name = "meta-llama/Meta-Llama-3-8B-Instruct"
model_name = "meta-llama/Llama-3.2-3B-Instruct"
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=quant_config,
)

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]c:\projects\synth-gen-llm\.venv\Lib\site-packages\bitsandbytes\backends\default\ops.py:223: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
c:\projects\synth-gen-llm\.venv\Lib\site-packages\bitsandbytes\backends\cpu\ops.py:36: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 254/254 [01:17<00:00,  3.27it/s]


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
def generate_dataset(dataset_type, number_of_data, max_new_tokens=3000):
    
    schemas = {
        "inventory": """
Return JSON list only.
Each row format:
{
 "item_name":"",
 "brand":"",
 "category":"",
 "color":"",
 "room":"",
 "container":"",
 "shelf":"",
 "owner":""
}
""",

        "search_queries": """
Return JSON list only.
Each row format:
{
 "query":"",
 "expected_item_name":""
}
Generate natural human search phrases.
""",

        "intent_commands": """
Return JSON list only.
Each row format:
{
 "input":"",
 "output":{
   "intent":"",
   "item_name":"",
   "room":"",
   "container":""
 }
}
""",

        "permissions": """
Return JSON list only.
Each row format:
{
 "user_id":"",
 "allowed_rooms":[]
}
""",

        "noisy_queries": """
Return JSON list only.
Each row format:
{
 "query":"",
 "intended_item":""
}
Include typo, shorthand, broken grammar.
"""
    }

    system_prompt = f"""
You are a professional synthetic dataset generator.

Generate {number_of_data} rows for dataset_type = {dataset_type}

{schemas[dataset_type]}

Rules:
- realistic household data
- no duplicates
- diverse rooms/items
- valid JSON only
- no markdown
"""

    messages = [
        {"role":"system","content":system_prompt},
        {"role":"user","content":"Generate the dataset by returning the dataset in JSON format only without any Markdown formatting."}
    ]

    input_ids = tokenizer.apply_chat_template(
        messages,
        # return_dict=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    attention_mask = torch.ones_like(input_ids, dtype=torch.long, device="cuda")


    streamer = TextStreamer(
        tokenizer, 
        # skip_prompt=True, 
        # skip_special_tokens=True
        )

    outputs = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=max_new_tokens,
        streamer=streamer
    )

    # return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
generate_dataset("inventory", 10)

In [1]:
import torch

if torch.cuda.is_available():
    print(f"CUDA is available! Number of GPUs: {torch.cuda.device_count()}")
    print(f"Current Device: {torch.cuda.current_device()}")
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
else:
    print("CUDA is not available. Using CPU instead.")


CUDA is available! Number of GPUs: 1
Current Device: 0
Device Name: NVIDIA GeForce RTX 5080
